# Document Analysis

In [2]:
import sys
import pandas as pd
import pyodbc

print('Python executable:', sys.executable)
print('Python version:', sys.version)
print('pandas version:', pd.__version__)
print('pyodbc version:', pyodbc.version)


Python executable: c:\Program Files\Python314\python.exe
Python version: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
pandas version: 3.0.3
pyodbc version: 5.3.0


## Cache Folder Path

In [3]:
from pathlib import Path
cache_folder = Path( r"C:\Users\s7909996\Downloads\E2X\temp")
cache_folder.mkdir(parents=True, exist_ok=True)
# doc_linked_multiple_entities.to_csv(cache_folder/"doc_linked_multiple_entities.csv",index=False)

## Fenergo PROD SQL Login
Use this if you connect in SSMS with **SQL Server Authentication**.

For security, enter password at runtime instead of saving it in the notebook.

In [4]:
import pyodbc

fen_conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=wvdbsp01160.bns.bns,5150;"
    "DATABASE=FenergoData;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

print("Connected")

Connected


## iManage PROD SQL Login

In [5]:
## Imanage PROD SQL Connect
server = "wvdbsp02291.bns.bns,5150"
database = "WS_GCMD"

im_conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    # f"UID={username};"
    # f"PWD={password};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)


print("Connected successfully")

Connected successfully


## Fenergo ALL DOCs

In [ ]:
sql_fen_docs_all = """
WITH OnboardedLE AS (  
SELECT DISTINCT LegalEntityId, max(caseLastUpdatedDate) as COBCompletedClosedDate  
FROM [FenergoData].[dbo].[vwCasesGrid] WITH (NOLOCK)  
WHERE caseTypeName = 'Client Onboarding' AND caseStatusName = 'Complete' AND maintenanceStatusName = 'Closed' group by legalentityid  
  

),  

OffboardedLE AS( SELECT DISTINCT lea.LegalEntityId, MAX(lea.lastUpdatedDate) as OffboardedDate FROM dbo.LEAssociate lea WITH (NOLOCK) LEFT JOIN dbo.LookupLEAssociateType leat WITH (NOLOCK) ON leat.Id = lea.LEAssociateTypeID LEFT JOIN dbo.LuLerolestatus lers WITH (NOLOCK) ON lers.Id = lea.LegalEntityRoleStatusId WHERE leat.Name = 'Client/Counterparty' and lers.Name ='Offboarded' /*and lea.LegalEntityId=27576 */ group by lea.legalentityid ), 

Jurisdiction as ( SELECT distinct [EntityId] ,[jurisdictionId],lv.[Name] FROM [FenergoData].[dbo].[Entityjurisdiction] ej inner join dbo.LegalEntity le on le.id= ej.EntityId left join dbo.jurisdiction lv on lv.Id= ej.[jurisdictionId] where ej.EntityTypeId=30 /*added to filter the legal entity type */group by [EntityId],jurisdictionId,lv.Name ), 

Jurisdictions as ( select distinct [EntityId] as LegalEntityId, STRING_AGG( Name, '|') as 'Jurisdictions' From Jurisdiction group by [EntityId] ), 

GlobalRisk AS ( SELECT LegalEntityID, RiskStatus as GlobalRisk FROM [FenergoData].[dbo].[vwLECompRiskProfile] WITH (NOLOCK) ), 


DocToLE AS ( /* 1) Documents linked directly to Legal Entity (BE = 30) */ 

SELECT DISTINCT lde.DocumentId, lde.BusinessEntityId AS LinkedBusinessEntityId, lde.EntityId AS LinkedEntityId,lde.EntityId AS LegalEntityId  
FROM LinkDocumentEntity lde WITH (NOLOCK)  
WHERE lde.BusinessEntityId = 30   
 
UNION    
 
/* 2) Documents linked to Case (BE = 1) -> map Case to Legal Entity via LegalEntityAssociation  */ 
 
SELECT DISTINCT lde.DocumentId, lde.BusinessEntityId AS LinkedBusinessEntityId, lde.EntityId AS LinkedEntityId,  lea.LegalEntityId AS LegalEntityId  
FROM LinkDocumentEntity lde WITH (NOLOCK) left JOIN LegalEntityAssociation lea WITH (NOLOCK)  
ON lea.EntityId = lde.EntityId and lea.BusinessEntityId = 1  
where lde.BusinessEntityId=1 AND lea.LegalEntityId IS NOT NULL  
  

),  

DocToLE_Dedup AS ( /* Deduplicate final mapping (cheap) */ 
SELECT DocumentId,  LegalEntityId FROM DocToLE GROUP BY DocumentId,  LegalEntityId  

) ,
FinalRaw as (
SELECT le.Id AS LegalEntityId, le.Name AS LegalEntityName, letp.Name AS LeType, d.Id AS DocumentId, d.Name AS DocumentName, d.Location as DocLink, /* Extract iManage document number from Location when present */ 
CASE WHEN d.Location IS NULL OR d.Location not like '%document%' THEN 'InvalidDocLink' ELSE CAST( TRY_CONVERT(INT, CASE WHEN d.Location LIKE '%!document:%' THEN SUBSTRING( d.Location, CHARINDEX('!document:', d.Location) + LEN('!document:'), CHARINDEX(',', d.Location) - (CHARINDEX('!document:', d.Location) + LEN('!document:')) ) ELSE NULL END ) AS VARCHAR(50) ) END AS iManage_Doc_Num, 

CASE WHEN d.Location IS NULL OR d.Location not like '%document%' THEN 'InvalidDocLink' ELSE CAST( TRY_CONVERT(int, CASE WHEN d.Location LIKE '%!document:%' AND CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) > 0 AND CHARINDEX(':', d.Location, CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1) > 0 THEN SUBSTRING( d.Location, CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1, CHARINDEX(':', d.Location, CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1) - (CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1) ) ELSE NULL END ) AS VARCHAR(50) )END AS iManage_Doc_Version, 

le.ReferenceId, dstatus.Name AS DocumentStatus, dtype.Name AS DocType, ddir.Name AS DocDirection, dpurpose.Name AS DocPurpose, dcat.Name AS DocCategory, d.LastUpdatedDate, d.LastUpdatedBy, d.CreatedDate, d.CreatedBy, case when c.legalentityid is not null then 'Onboarded' else 'No Completed closed COB' end as ClientOnboardingStatus, c.COBCompletedClosedDate, case when offb.legalentityid is not null then 'offboarded' else 'Not offboarded' end as ClientOffboardedStatus, offb.OffboardedDate, gr.GlobalRisk, j.Jurisdictions 
FROM    LegalEntity le WITH (NOLOCK) LEFT JOIN DocToLE_Dedup m  ON le.Id = m.LegalEntityId 

LEFT JOIN Document d WITH (NOLOCK) ON d.Id = m.DocumentId LEFT JOIN OnboardedLE c WITH (NOLOCK) ON c.Legalentityid=le.id LEFT JOIN OffboardedLE offb WITH (NOLOCK) ON offb.Legalentityid=le.id LEFT JOIN GlobalRisk gr WITH (NOLOCK) ON gr.Legalentityid=le.id LEFT JOIN Jurisdictions j WITH (NOLOCK) ON j.LegalEntityId=le.id LEFT JOIN LookupDocumentStatus dstatus WITH (NOLOCK) ON dstatus.Id = d.LookupDocumentStatusId LEFT JOIN DocumentType dtype WITH (NOLOCK) ON dtype.Id = d.DocumentTypeId LEFT JOIN LookupDocumentDirection ddir WITH (NOLOCK) ON ddir.Id = d.LookupDocumentDirectionId LEFT JOIN DocumentPurpose dpurpose WITH (NOLOCK) ON dpurpose.Id = d.DocumentPurposeId LEFT JOIN DocumentCategory dcat WITH (NOLOCK) ON dcat.Id = d.DocumentCategoryId LEFT JOIN LuLeSubTp letp WITH (NOLOCK) ON letp.Id = le.LegalEntitySubtypeId /* approximate filter docs only for imanage */ 

),

/* Final ranked */ 

FinalRanked as (
select *,
row_number() over(
partition by DocumentId order by 
LastUpdatedDate desc,
LegalEntityId desc



)as rn
from FinalRaw
)

/* Final query */ 
SELECT  * from FinalRanked where rn=1 --and LegalEntityId=1699971
ORDER BY LegalEntityId ASC, DocumentId DESC; """

fen_doc_all = pd.read_sql(sql_fen_docs_all,fen_conn)

fen_doc_all.to_csv(cache_folder/"fen_doc_all.csv",index=False)

## Fenergo Documents linked with multiple Entities

In [ ]:
doc_linked_multiple_entities= pd.read_sql("""

with DocToLE AS ( /* 1) Documents linked directly to Legal Entity: BusinessEntityId = 30 */ 
SELECT DISTINCT lde.DocumentId, lde.EntityId AS LegalEntityId 
FROM dbo.LinkDocumentEntity lde WITH (NOLOCK) 
WHERE lde.BusinessEntityId = 30 AND lde.DocumentId IS NOT NULL 

UNION  

/* 2) Documents linked to Case: BusinessEntityId = 1 Then map Case back to Legal Entity through LegalEntityAssociation */  

SELECT DISTINCT lde.DocumentId, lea.LegalEntityId 
FROM dbo.LinkDocumentEntity lde WITH (NOLOCK) 
INNER JOIN dbo.LegalEntityAssociation lea WITH (NOLOCK) ON lea.EntityId = lde.EntityId AND lea.BusinessEntityId = 1 
WHERE lde.BusinessEntityId = 1 AND lde.DocumentId IS NOT NULL AND lea.LegalEntityId IS NOT NULL ),  

DupDocs AS( 
select DocumentId from DocToLE GROUP BY DocumentId having count(distinct legalentityid)>1 
) 

select --count (distinct d.id)--
d.Id AS DocumentId, d.Name AS DocumentName, d.Location as DocLink, 
CASE WHEN d.Location IS NULL OR d.Location not like '%document%' THEN 'InvalidDocLink' ELSE CAST( TRY_CONVERT(INT, CASE WHEN d.Location LIKE '%!document:%' THEN SUBSTRING( d.Location, CHARINDEX('!document:', d.Location) + LEN('!document:'), CHARINDEX(',', d.Location) - (CHARINDEX('!document:', d.Location) + LEN('!document:')) ) ELSE NULL END ) AS VARCHAR(50) ) END AS iManage_Doc_Num,  
CASE WHEN d.Location IS NULL OR d.Location not like '%document%' THEN 'InvalidDocLink' ELSE CAST( TRY_CONVERT(int, CASE WHEN d.Location LIKE '%!document:%' AND CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) > 0 AND CHARINDEX(':', d.Location, CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1) > 0 THEN SUBSTRING( d.Location, CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1, CHARINDEX(':', d.Location, CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1) - (CHARINDEX(',', d.Location, CHARINDEX('!document:', d.Location)) + 1) ) ELSE NULL END ) AS VARCHAR(50) )END AS iManage_Doc_Version,  
CASE WHEN d.Location IS NULL  THEN 0 ELSE 1 End as IsValidDocLink,
m.LegalEntityId, le.Name AS LegalEntityName, letp.Name AS LeType, le.ReferenceId, 
dstatus.Name AS DocumentStatus, dtype.Name AS DocType, ddir.Name AS DocDirection, dpurpose.Name AS DocPurpose, dcat.Name AS DocCategory,
d.LastUpdatedDate, d.LastUpdatedBy, d.CreatedDate, d.CreatedBy

from DupDocs dd 
inner join DocToLE m on dd.DocumentId =m.DocumentId 
LEFT JOIN Document d WITH (NOLOCK) ON d.Id = m.DocumentId 
LEFT JOIN LegalEntity le WITH (NOLOCK) ON le.Id = m.LegalEntityId 
LEFT JOIN LookupDocumentStatus dstatus WITH (NOLOCK) ON dstatus.Id = d.LookupDocumentStatusId 
LEFT JOIN DocumentType dtype WITH (NOLOCK) ON dtype.Id = d.DocumentTypeId 
LEFT JOIN LookupDocumentDirection ddir WITH (NOLOCK) ON ddir.Id = d.LookupDocumentDirectionId 
LEFT JOIN DocumentPurpose dpurpose WITH (NOLOCK) ON dpurpose.Id = d.DocumentPurposeId 
LEFT JOIN DocumentCategory dcat WITH (NOLOCK) ON dcat.Id = d.DocumentCategoryId 
LEFT JOIN LuLeSubTp letp WITH (NOLOCK) ON letp.Id = le.LegalEntitySubtypeId 
ORDER BY d.Id, le.Id;

""",fen_conn,)


## IM KYC Workspace Docs

In [30]:
sql_im_kyc_ws=r"""
--document list in iMange that from KYC by woekspace name  

with workspace_level as  
( 
select prj_id, prj_name, prj_owner, tree_id, editwhen--,subtype 
from mhgroup.projects 
where 
prj_id = tree_id 
AND subtype = 'work' 
AND upper(prj_name) LIKE '%KYC ONBOARDING%' 
) 

select  distinct * from (  
Select 
w.prj_name as 'Workspace',
dm.docnum, 
dm.version, 
dm.docname, 
dm.c1alias,c1.C_DESCRIPT, 
dm.c2alias, 
dm.author, 
dm.t_alias, 
dm.subclass_alias, 
dm.[type], 
dm.docsize, 
dm.entrywhen, 
dm.editwhen, 
dm.editprofilewhen, 
dm.fileentrywhen, 
dm.fileeditwhen, 
dm.docloc, 
dm.Operator, 
ROW_NUMBER() OVER (PARTITION BY dm.docnum ORDER BY dm.version DESC) AS rn 
from mhgroup.projects p WITH (NOLOCK) 
INNER join workspace_level W WITH (NOLOCK) on w.prj_id = p.tree_id 
INNER join mhgroup.project_items b WITH (NOLOCK) on b.PRJ_ID = p.PRJ_ID 
INNER join mhgroup.docmaster dm WITH (NOLOCK) on b.item_id = dm.docnum 
LEFT join MHGROUP.CUSTOM1 c1 WITH (NOLOCK) on dm.C1ALIAS = c1.CUSTOM_ALIAS 
WHERE dm.[type] IN( 'D' ) 
) x 

WHERE x.rn = 1  
Order by x.c1alias asc, x.docnum asc """
im_kyc_ws = pd.read_sql(sql_im_kyc_ws, im_conn)
im_kyc_ws.to_csv(cache_folder/"im_kyc_ws.csv",index=False)

C:\Users\s7909996\AppData\Local\Temp\ipykernel_6928\2056632497.py:46: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  im_kyc_ws = pd.read_sql(sql_im_kyc_ws, im_conn)


## IM KYC Client Docs

In [ ]:
sql_im_kyc_client=pd.read_sql("""
/* Query 2 simplified:
   KYC documents based on client name / C2Alias
   Only keep documents that are NOT already in KYC workspace list
*/

;WITH C2Alias_Base AS
(
    SELECT
        dm.docnum,
        dm.version,
        dm.docname,
        dm.c1alias,
        dm.c2alias,
        dm.author,
        dm.t_alias,
        dm.subclass_alias,
        dm.[type],
        dm.docsize,
        dm.entrywhen,
        dm.editwhen,
        dm.editprofilewhen,
        dm.fileentrywhen,
        dm.fileeditwhen,
        dm.docloc,
        dm.Operator,
        ROW_NUMBER() OVER
        (
            PARTITION BY dm.docnum
            ORDER BY dm.version DESC
        ) AS rn
    FROM mhgroup.docmaster dm WITH (NOLOCK)
    WHERE UPPER(dm.c2alias) LIKE '%KYC_ONBOARDING%'
      AND dm.[type] = 'D'
),

C2Alias_KYC AS
(
    SELECT
        b.docnum,
        b.version,
        b.docname,
        b.c1alias,
        c1.C_DESCRIPT,
        b.c2alias,
        b.author,
        b.t_alias,
        b.subclass_alias,
        b.[type],
        b.docsize,
        b.entrywhen,
        b.editwhen,
        b.editprofilewhen,
        b.fileentrywhen,
        b.fileeditwhen,
        b.docloc,
        b.Operator
    FROM C2Alias_Base b
    LEFT JOIN mhgroup.CUSTOM1 c1 WITH (NOLOCK)
        ON b.c1alias = c1.CUSTOM_ALIAS
    WHERE b.rn = 1
),

Workspace_KYC_Docs AS
(
    SELECT DISTINCT
        pi.item_id AS docnum
    FROM mhgroup.projects w WITH (NOLOCK)
    INNER JOIN mhgroup.projects p WITH (NOLOCK)
        ON p.tree_id = w.prj_id
    INNER JOIN mhgroup.project_items pi WITH (NOLOCK)
        ON pi.prj_id = p.prj_id
    INNER JOIN mhgroup.docmaster dm WITH (NOLOCK)
        ON dm.docnum = pi.item_id
       AND dm.[type] = 'D'
    WHERE w.prj_id = w.tree_id
      AND w.subtype = 'work'
      AND UPPER(w.prj_name) LIKE '%KYC ONBOARDING%'
      AND pi.item_id IS NOT NULL
),

Workspace_Name AS
(
    SELECT
        pi.item_id AS docnum,
        w.prj_name AS workspace_name,
        ROW_NUMBER() OVER
        (
            PARTITION BY pi.item_id
            ORDER BY p.editwhen DESC
        ) AS rn
    FROM mhgroup.project_items pi WITH (NOLOCK)
    INNER JOIN mhgroup.projects p WITH (NOLOCK)
        ON pi.prj_id = p.prj_id
    INNER JOIN mhgroup.projects w WITH (NOLOCK)
        ON p.tree_id = w.prj_id
       AND w.subtype = 'work'
       AND w.prj_id = w.tree_id
)

SELECT
  
    wn.workspace_name,
    c.docnum,
    c.version,
    c.docname,
    c.c1alias,
    c.C_DESCRIPT,
    c.c2alias,
    c.author,
    c.t_alias,
    c.subclass_alias,
    c.[type],
    c.docsize,
    c.entrywhen,
    c.editwhen,
    c.editprofilewhen,
    c.fileentrywhen,
    c.fileeditwhen,
    c.docloc,
    c.Operator
FROM C2Alias_KYC c
LEFT JOIN Workspace_Name wn
    ON c.docnum = wn.docnum
   AND wn.rn = 1
WHERE NOT EXISTS
(
    SELECT 1
    FROM Workspace_KYC_Docs w
    WHERE w.docnum = c.docnum
)
ORDER BY
    c.c1alias,
    c.docnum;  """,im_conn)


im_kyc_client = pd.read_sql(sql_im_kyc_client, im_conn)
im_kyc_client .to_csv(cache_folder/"im_kyc_client.csv",index=False)

C:\Users\s7909996\AppData\Local\Temp\ipykernel_6928\3310020903.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sql_im_kyc_client=pd.read_sql("""


# Start from here -- Cached on July 14th

## Read all Fenergo Docs

In [11]:
fen_doc_all=pd.read_csv(cache_folder/"fen_doc_all.csv")
len(fen_doc_all)

C:\Users\s7909996\AppData\Local\Temp\ipykernel_6928\3790001465.py:1: DtypeWarning: Columns (0: ReferenceId) have mixed types. Specify dtype option on import or set low_memory=False.
  fen_doc_all=pd.read_csv(cache_folder/"fen_doc_all.csv")


## Fen Doc linked with multiple FIDs

In [32]:
doc_linked_multiple_entities = pd.read_csv(cache_folder/"doc_linked_multiple_entities.csv", low_memory=False)
doc_linked_multiple_entities.shape

(22055, 19)

## Read IM KYC WS DOC

In [31]:
im_kyc_ws=pd.read_csv(cache_folder/"im_kyc_ws.csv")
im_kyc_ws.shape

C:\Users\s7909996\AppData\Local\Temp\ipykernel_6928\3955034721.py:1: DtypeWarning: Columns (0: c1alias) have mixed types. Specify dtype option on import or set low_memory=False.
  im_kyc_ws=pd.read_csv(cache_folder/"im_kyc_ws.csv")


(1868798, 20)

## Read IM KYC Client DOC

In [51]:
im_kyc_client=pd.read_csv(cache_folder/"im_kyc_client.csv")
im_kyc_client.shape

(6609, 19)

## Merged IM KYC DOC

In [69]:
im_kyc_merge = (
    pd.concat(
        [im_kyc_ws, im_kyc_client],
        ignore_index=True
    )
    .sort_values(
        ["c1alias","docnum", "version"],
        ascending=[True,True, False]
    )
    .drop_duplicates(
        subset=["c1alias","docnum"],
        keep="first"      # keep latest version
    )
    .reset_index(drop=True)
)

print("Workspace docs :", im_kyc_ws["docnum"].nunique())
print("Client docs    :", im_kyc_client["docnum"].nunique())
print("Merged docs    :", im_kyc_merge["docnum"].nunique())


Workspace docs : 1868798
Client docs    : 6609
Merged docs    : 1875407


# Analysis

In [70]:
print(im_kyc_merge.columns)
print(fen_doc_all.columns)

Index(['Workspace', 'docnum', 'version', 'docname', 'c1alias', 'C_DESCRIPT',
       'c2alias', 'author', 't_alias', 'subclass_alias', 'type', 'docsize',
       'entrywhen', 'editwhen', 'editprofilewhen', 'fileentrywhen',
       'fileeditwhen', 'docloc', 'Operator', 'rn', 'workspace_name'],
      dtype='str')
Index(['LegalEntityId', 'LegalEntityName', 'LeType', 'DocumentId',
       'DocumentName', 'DocLink', 'iManage_Doc_Num', 'iManage_Doc_Version',
       'ReferenceId', 'DocumentStatus', 'DocType', 'DocDirection',
       'DocPurpose', 'DocCategory', 'LastUpdatedDate', 'LastUpdatedBy',
       'CreatedDate', 'CreatedBy', 'ClientOnboardingStatus',
       'COBCompletedClosedDate', 'ClientOffboardedStatus', 'OffboardedDate',
       'GlobalRisk', 'Jurisdictions', 'rn'],
      dtype='str')


In [73]:
import pandas as pd
import numpy as np
from pathlib import Path

# =====================================================
# 1. Copy source data
# =====================================================

fen = fen_doc_all.copy()
im = im_kyc_merge.copy()

# Preserve original Fenergo row order
fen["_fen_row_id"] = range(len(fen))
im["_im_row_id"] = range(len(im))

# Store original iManage columns before adding helper columns
im_original_cols = im_kyc_merge.columns.tolist()

# =====================================================
# 2. Helper functions
# =====================================================

def clean_id(s):
    """
    For numeric/string IDs:
    - convert to string
    - remove trailing .0
    - trim
    - uppercase
    - convert blank / nan-like text to NA
    """
    out = (
        s.astype("string")
         .str.replace(".0", "", regex=False)
         .str.strip()
         .str.upper()
    )

    out = out.replace(
        {
            "": pd.NA,
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "<NA>": pd.NA
        }
    )

    return out


def clean_text(s):
    """
    For names/document names:
    - convert to string
    - trim
    - uppercase
    - collapse multiple spaces
    - convert blank / nan-like text to NA
    """
    out = (
        s.astype("string")
         .str.strip()
         .str.upper()
         .str.replace(r"\s+", " ", regex=True)
    )

    out = out.replace(
        {
            "": pd.NA,
            "NAN": pd.NA,
            "NONE": pd.NA,
            "NULL": pd.NA,
            "<NA>": pd.NA
        }
    )

    return out


# =====================================================
# 3. Normalize Fenergo keys
# =====================================================

# Layer 1: iManage_Doc_Num = docnum
fen["iManage_Doc_Num_txt"] = clean_id(fen["iManage_Doc_Num"])

# Layer 2: LegalEntityId + DocumentName
fen["FID_txt"] = clean_id(fen["LegalEntityId"])

# Layer 3: ReferenceId + DocumentName
# ReferenceId is text, so preserve as text but normalize case/spaces
fen["ReferenceId_txt"] = clean_id(fen["ReferenceId"])

# Layer 4: LegalEntityName + DocumentName
fen["LegalEntityName_txt"] = clean_text(fen["LegalEntityName"])

# Shared document name key
fen["DocumentName_txt"] = clean_text(fen["DocumentName"])

# =====================================================
# 4. Normalize iManage keys
# =====================================================

im["docnum_txt"] = clean_id(im["docnum"])
im["c1alias_txt"] = clean_id(im["c1alias"])
im["C_DESCRIPT_txt"] = clean_text(im["C_DESCRIPT"])
im["docname_txt"] = clean_text(im["docname"])

# =====================================================
# 5. Create composite keys
# =====================================================

# Layer 2:
# Fenergo LegalEntityId + DocumentName
# iManage c1alias + docname
fen["L2_KEY"] = fen["FID_txt"].fillna("") + "||" + fen["DocumentName_txt"].fillna("")
im["L2_KEY"] = im["c1alias_txt"].fillna("") + "||" + im["docname_txt"].fillna("")

# Layer 3:
# Fenergo ReferenceId + DocumentName
# iManage c1alias + docname
fen["L3_KEY"] = fen["ReferenceId_txt"].fillna("") + "||" + fen["DocumentName_txt"].fillna("")
im["L3_KEY"] = im["c1alias_txt"].fillna("") + "||" + im["docname_txt"].fillna("")

# Layer 4:
# Fenergo LegalEntityName + DocumentName
# iManage C_DESCRIPT + docname
fen["L4_KEY"] = fen["LegalEntityName_txt"].fillna("") + "||" + fen["DocumentName_txt"].fillna("")
im["L4_KEY"] = im["C_DESCRIPT_txt"].fillna("") + "||" + im["docname_txt"].fillna("")

# Convert invalid composite keys to NA
fen.loc[fen["L2_KEY"].eq("||"), "L2_KEY"] = pd.NA
fen.loc[fen["L3_KEY"].eq("||"), "L3_KEY"] = pd.NA
fen.loc[fen["L4_KEY"].eq("||"), "L4_KEY"] = pd.NA

im.loc[im["L2_KEY"].eq("||"), "L2_KEY"] = pd.NA
im.loc[im["L3_KEY"].eq("||"), "L3_KEY"] = pd.NA
im.loc[im["L4_KEY"].eq("||"), "L4_KEY"] = pd.NA

# =====================================================
# 6. Build iManage lookup tables
# Version ignored because im_kyc_merge already has max version
# =====================================================

# Layer 1 lookup: one row per docnum
# This prevents row multiplication when same docnum appears multiple times.
im_l1_lookup = (
    im[im["docnum_txt"].notna()]
    .sort_values(["docnum_txt", "_im_row_id"])
    .drop_duplicates(subset=["docnum_txt"], keep="first")
)

# Layer 2 lookup: one row per c1alias + docname
im_l2_lookup = (
    im[im["L2_KEY"].notna()]
    .sort_values(["L2_KEY", "_im_row_id"])
    .drop_duplicates(subset=["L2_KEY"], keep="first")
)

# Layer 3 lookup: same iManage-side key as Layer 2,
# but matched against Fenergo ReferenceId instead of LegalEntityId.
im_l3_lookup = (
    im[im["L3_KEY"].notna()]
    .sort_values(["L3_KEY", "_im_row_id"])
    .drop_duplicates(subset=["L3_KEY"], keep="first")
)

# Layer 4 lookup: one row per C_DESCRIPT + docname
im_l4_lookup = (
    im[im["L4_KEY"].notna()]
    .sort_values(["L4_KEY", "_im_row_id"])
    .drop_duplicates(subset=["L4_KEY"], keep="first")
)

# =====================================================
# 7. Prefix iManage lookup columns
# =====================================================

# Keep original iManage columns plus required keys
im_l1_lookup = (
    im_l1_lookup[im_original_cols + ["docnum_txt"]]
    .add_prefix("im1_")
)

im_l2_lookup = (
    im_l2_lookup[im_original_cols + ["L2_KEY"]]
    .add_prefix("im2_")
)

im_l3_lookup = (
    im_l3_lookup[im_original_cols + ["L3_KEY"]]
    .add_prefix("im3_")
)

im_l4_lookup = (
    im_l4_lookup[im_original_cols + ["L4_KEY"]]
    .add_prefix("im4_")
)

# =====================================================
# 8. Four left merges by priority
# =====================================================

# Layer 1: iManage_Doc_Num = docnum
m1 = fen.merge(
    im_l1_lookup,
    how="left",
    left_on="iManage_Doc_Num_txt",
    right_on="im1_docnum_txt"
)

# Layer 2: LegalEntityId + DocumentName = c1alias + docname
m2 = fen.merge(
    im_l2_lookup,
    how="left",
    left_on="L2_KEY",
    right_on="im2_L2_KEY"
)

# Layer 3: ReferenceId + DocumentName = c1alias + docname
m3 = fen.merge(
    im_l3_lookup,
    how="left",
    left_on="L3_KEY",
    right_on="im3_L3_KEY"
)

# Layer 4: LegalEntityName + DocumentName = C_DESCRIPT + docname
m4 = fen.merge(
    im_l4_lookup,
    how="left",
    left_on="L4_KEY",
    right_on="im4_L4_KEY"
)

# =====================================================
# 9. Safety check: each merge should preserve row count
# =====================================================

print("fen rows:", len(fen))
print("m1 rows :", len(m1))
print("m2 rows :", len(m2))
print("m3 rows :", len(m3))
print("m4 rows :", len(m4))

if not (
    len(fen) == len(m1) == len(m2) == len(m3) == len(m4)
):
    raise ValueError(
        "One of the merge layers multiplied rows. Check duplicate keys in lookup tables."
    )

# =====================================================
# 10. Build final dataframe
# =====================================================

final = fen.copy()

# Bring every original iManage column back with im_ prefix,
# using priority m1 -> m2 -> m3 -> m4.
for col in im_original_cols:
    final[f"im_{col}"] = (
        m1[f"im1_{col}"]
        .combine_first(m2[f"im2_{col}"])
        .combine_first(m3[f"im3_{col}"])
        .combine_first(m4[f"im4_{col}"])
    )

# =====================================================
# 11. Match indicators
# =====================================================

match1 = m1["im1_docnum"].notna()

match2 = (
    ~match1
    & m2["im2_docnum"].notna()
)

match3 = (
    ~match1
    & ~match2
    & m3["im3_docnum"].notna()
)

match4 = (
    ~match1
    & ~match2
    & ~match3
    & m4["im4_docnum"].notna()
)

final["MatchBoolean"] = np.select(
    [match1, match2, match3, match4],
    [1, 1, 1, 1],
    default=0
).astype(int)

final["MatchedBy"] = np.select(
    [match1, match2, match3, match4],
    [
        "Matched By DocNum",
        "Matched By FID+DocName",
        "Matched By RefID+DocName",
        "Matched By ClientName+DocName"
    ],
    default="Not Matched"
)

# Optional QA flags
final["MatchedDocNum"] = match1.astype(int)
final["MatchedFIDDocName"] = match2.astype(int)
final["MatchedRefIdDocName"] = match3.astype(int)
final["MatchedClientNameDocName"] = match4.astype(int)

# =====================================================
# 12. Remove helper columns from final
# =====================================================

helper_cols = [
    "_fen_row_id",
    "iManage_Doc_Num_txt",
    "FID_txt",
    "ReferenceId_txt",
    "LegalEntityName_txt",
    "DocumentName_txt",
    "L2_KEY",
    "L3_KEY",
    "L4_KEY"
]

final = final.drop(
    columns=[c for c in helper_cols if c in final.columns],
    errors="ignore"
)

# =====================================================
# 13. Validation summary
# =====================================================

print("\nOriginal fen_doc_all rows :", len(fen_doc_all))
print("Final rows                :", len(final))
print("Matched rows              :", final["MatchBoolean"].sum())
print("Not matched rows          :", (final["MatchBoolean"] == 0).sum())

print("\nMatch Summary:")
print(final["MatchedBy"].value_counts(dropna=False))

print("\nFinal shape:", final.shape)

display(final.head())

MemoryError: Unable to allocate 14.3 MiB for an array with shape (1875407,) and data type object